# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset on clinicopathological and molecular features of second primary colorectal cancer in cancer survivors, using the `mlcroissant` library.

### Dataset Source

The dataset is defined by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset metadata and object
dataset = mlc.Dataset(croissant_url)
# Display name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

Review available record sets and fields. All entities are referenced by their `@id`.

Below, we list all record sets and their field `@id`s.

In [ ]:
# List all available record sets and their fields by @id
print("Available record sets:")
for rs in dataset.record_sets:
    print(f"- Record set @id: {rs.id} (name: {rs.name})")
    if hasattr(rs, 'fields'):
        for fld in rs.fields:
            print(f"    - Field @id: {fld.id} (name: {fld.name})")

## 3. Data Extraction

Load data from the main record set(s) into pandas DataFrame(s) for analysis. Use the record set and field `@id`s.

**Note:** For this dataset, the main clinical data table is typically named `second_primary_colorectal_cancer_patients` or similar (see printed record set IDs above for the exact value). For demonstration, we'll load all available record sets.

In [ ]:
# Gather all record set @id values
all_record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for record_set_id in all_record_set_ids:
    print(f"\nLoading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records with columns: {list(df.columns)}")

Let's inspect the main clinical record set (`@id`). Replace with the actual @id of your main clinical data table found above.

In [ ]:
# Select one record set for demonstration (update with actual main table @id)
main_record_set_id = all_record_set_ids[0]  # Replace with the correct one if needed
df = dataframes[main_record_set_id]
print(f"First few rows from record set @id: {main_record_set_id}")
df.head()

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps. Choose a numeric field (e.g., age, interval, or any numeric clinical variable) by its `@id` to demonstrate filtering and transformation. **First, list available numeric fields:**

In [ ]:
print("Columns and sample values in the main record set:")
for col in df.columns:
    print(f"- {col}: example -> {df[col].iloc[0] if not df.empty else 'N/A'}")

Let's assume the field `age_at_second_primary_diagnosis` exists, and its field `@id` in the schema is `https://api.app.sen.science/frontiers/7862866/field-age_at_second_primary_diagnosis` (please adjust using your data; you can look up the actual field @id from the earlier overview if different). We'll filter for patients older than 60, normalize age, and group by sex (assuming column exists with a field @id like `https://api.app.sen.science/frontiers/7862866/field-sex`).

In [ ]:
# Define the field (column) @id for the age variable
age_field_id = 'https://api.app.sen.science/frontiers/7862866/field-age_at_second_primary_diagnosis'  # update to match printed columns
# For demonstration, if this exact @id isn't present, fall back to first numeric column
if age_field_id not in df.columns:
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    if len(numeric_cols) > 0:
        age_field_id = numeric_cols[0]
print(f"Using numeric field @id: {age_field_id}")

# Filter for age > 60
threshold = 60
filtered_df = df[df[age_field_id] > threshold]
print(f"Filtered records with {age_field_id} > {threshold} (n={filtered_df.shape[0]}):")
display(filtered_df.head())

# Normalize age
filtered_df[f"{age_field_id}_normalized"] = (filtered_df[age_field_id] - filtered_df[age_field_id].mean()) / filtered_df[age_field_id].std()
print(f"Normalized {age_field_id} for filtered records:")
display(filtered_df[[age_field_id, f"{age_field_id}_normalized"]].head())

# Group by sex field (edit the @id if needed)
sex_field_id = 'https://api.app.sen.science/frontiers/7862866/field-sex'
if sex_field_id not in df.columns:
    alt_cats = df.select_dtypes(include='object').columns
    if len(alt_cats) > 0:
        sex_field_id = alt_cats[0]
if sex_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(sex_field_id)[age_field_id].mean()
    print(f"Grouped mean {age_field_id} by {sex_field_id}:")
    print(grouped_df)
else:
    print(f"Could not find grouping field {sex_field_id} in columns.")

## 5. Visualization

Visualize the distribution of age at diagnosis and the group statistics by sex.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Histogram of age at diagnosis
plt.figure(figsize=(8, 5))
sns.histplot(df[age_field_id].dropna(), bins=15, kde=True, color='royalblue')
plt.title('Distribution of Age at Second Primary Colorectal Cancer Diagnosis')
plt.xlabel('Age at Diagnosis')
plt.ylabel('Number of Patients')
plt.show()

# Boxplot of age by sex
if sex_field_id in df.columns:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=df[sex_field_id], y=df[age_field_id])
    plt.title('Age at Diagnosis by Sex')
    plt.xlabel('Sex')
    plt.ylabel('Age at Diagnosis')
    plt.show()
else:
    print(f"Sex field ({sex_field_id}) not present for grouping.")

## 6. Conclusion

In this notebook, we've demonstrated how to load, explore, and analyze a clinical cancer dataset defined using a Croissant schema and managed with the `mlcroissant` library. Key steps included extracting field and record set information by their `@id`, performing basic filtering and normalization of numeric data, grouping by categorical variables, and visualizing major trends.

This workflow can be adapted to new Croissant datasets by referencing schema entities by their `@id` and updating the processing steps accordingly.